In [ ]:
from keras.datasets import mnist
from keras.layers import Input, Dense
from keras.models import Model
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

#!pip install xgboost
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score, ConfusionMatrixDisplay
from sklearn.manifold import TSNE


In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

In [ ]:
X_train = X_train.astype('float32')/255
X_test = X_test.astype('float32')/255

In [ ]:
X_train.shape, X_test.shape

In [ ]:
X_train = X_train.reshape(len(X_train), np.prod(X_train.shape[1:]))
X_test = X_test.reshape(len(X_test), np.prod(X_test.shape[1:]))
print(X_train.shape)
print(X_test.shape)

In [ ]:
input_img= Input(shape=(784,))
encoded = Dense(units=8, activation='relu')(input_img)
decoded = Dense(units=784, activation='sigmoid')(encoded)

In [ ]:
autoencoder=Model(input_img, decoded)
autoencoder.summary()

In [ ]:
encoder = Model(input_img, encoded)
encoder.summary()


In [ ]:
autoencoder.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
autoencoder.fit(X_train, X_train,
                epochs=50,
                batch_size=256,
                shuffle=True,
                validation_data=(X_test, X_test))

In [ ]:
encoded_imgs = encoder.predict(X_test)
predicted = autoencoder.predict(X_test)

In [ ]:
encoded_imgs.shape

In [ ]:
predicted.shape

In [ ]:
plt.figure(figsize=(40, 4))
for i in range(10):
    # display original
    ax = plt.subplot(3, 20, i + 1)
    plt.imshow(X_test[i].reshape(28, 28))
    plt.gray()
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # display encoded image
    ax = plt.subplot(3, 20, i + 1 + 20)
    plt.imshow(encoded_imgs[i].reshape(2,4))
    plt.gray()
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    # display reconstruction
    ax = plt.subplot(3, 20, 2*20 +i+ 1)
    plt.imshow(predicted[i].reshape(28, 28))
    plt.gray()
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)



plt.show()

In [ ]:
X_train_encoded = encoder.predict(X_train)
X_test_encoded = encoder.predict(X_test)

In [ ]:
print(X_train_encoded.shape)
print(X_test_encoded.shape)

# Processo de Classificaçao com Random Forest

In [ ]:
modelRf = RandomForestClassifier()
modelRf.fit(X_train_encoded, y_train)
y_pred = modelRf.predict(X_test_encoded)

In [ ]:
print(classification_report(y_test, y_pred))

print(accuracy_score(y_test, y_pred))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

cm = confusion_matrix(y_test, y_pred, labels=modelRf.classes_, normalize = 'true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=modelRf.classes_)

disp.plot(ax = ax)
plt.show()

# Exemplo de Data Visualization a partir da camada Coded do Autoencoder

In [ ]:
tsne = TSNE(n_iter=2000)
X_embedded_tsne = tsne.fit_transform(X_test_encoded)


In [ ]:
sns.set(rc={'figure.figsize':(20,10)})
palette = sns.color_palette("bright", 10)
sns.scatterplot(x = X_embedded_tsne[:,0], y = X_embedded_tsne[:,1], hue=y_test, legend='full', palette=palette)

##PCA

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components=8)
X_train_pca = pca.fit_transform(X_train_encoded)
X_test_pca = pca.transform(X_test_encoded)

In [ ]:
modelRf = RandomForestClassifier()
modelRf.fit(X_train_pca, y_train)
y_pred = modelRf.predict(X_test_pca)

In [ ]:
print(classification_report(y_test, y_pred))

print(accuracy_score(y_test, y_pred))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

cm = confusion_matrix(y_test, y_pred, labels=modelRf.classes_, normalize = 'true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=modelRf.classes_)

disp.plot(ax = ax)
plt.show()